# Notebook 08 — Nested Ridge Modeling and Out-of-Fold Evaluation

**Project:** WR-MovementIQ  
**Purpose:** Compare traditional Combine measurements, tracking-derived Combine features, and their combination using leakage-safe nested cross-validation.

This notebook uses the frozen core common dataset and the validated outer receiver-grouped folds from Notebook 07. It does not assume tracking features are superior; it tests their incremental predictive value.

**Models**
- Baseline: training-fold median
- Traditional: `forty`, `vertical`, `broad_jump`
- Tracking: six tracking-derived Combine features
- Combined: all nine predictors

All model specifications use the same eligible rows for each outcome. Scaling is fitted inside each training fold. Ridge alpha is selected using inner GroupKFold CV grouped by receiver.

In [1]:
from pathlib import Path
import json, hashlib, warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "data").exists(), f"Project root not found from {Path.cwd()}"

DATA_PATH = PROJECT_ROOT / "data/features/wr_analysis_dataset_core_common.parquet"
FOLD_PATH = PROJECT_ROOT / "reports/modeling_design/outer_grouped_fold_assignments.parquet"
if not FOLD_PATH.exists():
    FOLD_PATH = PROJECT_ROOT / "reports/modeling_design/outer_grouped_fold_assignments.csv"

OUTPUT_DIR = PROJECT_ROOT / "reports/nested_ridge_modeling"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)
print("Fold assignments:", FOLD_PATH)
print("Output:", OUTPUT_DIR)

Project: D:\PROJECTS\WR-MovementIQ
Dataset: D:\PROJECTS\WR-MovementIQ\data\features\wr_analysis_dataset_core_common.parquet
Fold assignments: D:\PROJECTS\WR-MovementIQ\reports\modeling_design\outer_grouped_fold_assignments.parquet
Output: D:\PROJECTS\WR-MovementIQ\reports\nested_ridge_modeling


In [2]:
df = pd.read_parquet(DATA_PATH)
folds = pd.read_parquet(FOLD_PATH) if FOLD_PATH.suffix == ".parquet" else pd.read_csv(FOLD_PATH)

print("Dataset shape:", df.shape)
print("Fold columns:", folds.columns.tolist())
print("Dataset columns:", df.columns.tolist())
display(df.head())
display(folds.head())

Dataset shape: (93, 103)
Fold columns: ['draft_year', 'nfl_id', 'season', 'outer_fold']
Dataset columns: ['draft_year', 'season', 'nfl_id', 'game_mean_speed_raw_median', 'game_mean_speed_raw_mean', 'game_peak_speed_raw_median', 'game_peak_speed_raw_mean', 'game_p95_speed_raw_median', 'game_p95_speed_raw_mean', 'game_mean_acceleration_raw_median', 'game_mean_acceleration_raw_mean', 'game_peak_acceleration_raw_median', 'game_peak_acceleration_raw_mean', 'game_minimum_acceleration_raw_median', 'game_minimum_acceleration_raw_mean', 'game_summed_recorded_displacement_raw_median', 'game_summed_recorded_displacement_raw_mean', 'game_coordinate_path_length_raw_median', 'game_coordinate_path_length_raw_mean', 'game_total_direction_change_raw_median', 'game_total_direction_change_raw_mean', 'game_path_efficiency_median', 'game_path_efficiency_mean', 'game_snap_to_throw_seconds_median', 'game_snap_to_throw_seconds_mean', 'game_separation_at_pass_forward_median', 'game_separation_at_pass_forward_m

,draft_year,season,nfl_id,game_mean_speed_raw_median,game_mean_speed_raw_mean,game_peak_speed_raw_median,game_peak_speed_raw_mean,game_p95_speed_raw_median,game_p95_speed_raw_mean,game_mean_acceleration_raw_median,game_mean_acceleration_raw_mean,game_peak_acceleration_raw_median,game_peak_acceleration_raw_mean,game_minimum_acceleration_raw_median,game_minimum_acceleration_raw_mean,game_summed_recorded_displacement_raw_median,game_summed_recorded_displacement_raw_mean,game_coordinate_path_length_raw_median,game_coordinate_path_length_raw_mean,game_total_direction_change_raw_median,game_total_direction_change_raw_mean,game_path_efficiency_median,game_path_efficiency_mean,game_snap_to_throw_seconds_median,game_snap_to_throw_seconds_mean,game_separation_at_pass_forward_median,game_separation_at_pass_forward_mean,game_receiving_yards_median,game_receiving_yards_mean,game_tracking_observations_median,game_maximum_sampling_gap_seconds_median,game_maximum_sampling_gap_seconds_max,game_gaps_over_0_15_seconds_sum,game_rows_with_missing_movement_values_sum,receiver_plays,games,route_types_observed,plays_with_receiving_yards,plays_with_separation,receiving_yards_coverage_pct,separation_coverage_pct,primary_drills_observed,primary_attempts_observed,combine_mean_speed_median_across_drills,combine_mean_speed_iqr_across_drills,combine_peak_speed_median_across_drills,combine_peak_speed_iqr_across_drills,combine_p95_speed_median_across_drills,combine_p95_speed_iqr_across_drills,combine_mean_acceleration_median_across_drills,...,combine_coordinate_path_length_median_across_drills,combine_coordinate_path_length_iqr_across_drills,combine_net_displacement_median_across_drills,combine_net_displacement_iqr_across_drills,combine_path_efficiency_median_across_drills,combine_path_efficiency_iqr_across_drills,combine_coordinate_total_turn_median_across_drills,combine_coordinate_total_turn_iqr_across_drills,combine_coordinate_mean_turn_median_across_drills,combine_coordinate_mean_turn_iqr_across_drills,combine_coordinate_p95_turn_median_across_drills,combine_coordinate_p95_turn_iqr_across_drills,combine_turning_rate_degrees_per_second_median_across_drills,combine_turning_rate_degrees_per_second_iqr_across_drills,median_turning_pair_coverage,drills_with_turning_coverage_below_70_pct,nfl_position,combine_position,combine_height,combine_weight,hand_size,arm_length,wing_span,ten_yd_split,forty,vertical,broad_jump,three_cone,short_shuttle,bench_reps,ngs_athleticism_score,ngs_college_production_score,ngs_final_score,combine_position_results,combine_height_results,combine_weight_results,hand_size_results,arm_length_results,wing_span_results,ten_yd_split_results,forty_results,vertical_results,broad_jump_results,three_cone_results,short_shuttle_results,bench_reps_results,ngs_athleticism_score_results,ngs_college_production_score_results,ngs_final_score_results,complete_all_model_features
0,2023,2023,55887,4.529600,4.361201,7.140,6.841748,7.08400,6.748447,2.870769,2.930558,5.410,5.454078,0.01,0.216117,13.74,14.272039,13.687475,14.228596,139.03,150.521942,0.966198,0.927158,3.0,3.064078,3.297347,3.599274,10.0,12.769231,31.0,0.1,0.1,0,0,103,17,10,65,103,63.106796,100.0,5,6,6.080678,1.346808,8.416667,0.350000,8.207333,0.359333,3.136228,...,35.542164,12.427440,35.426998,14.552285,0.888016,0.075139,162.730758,102.127300,2.506501,0.206404,6.896084,1.994038,23.572984,5.931670,0.953125,0,WR,WR,71.125,173,8.750,30.875,74.500,1.56,4.49,34.0,122.0,NaN,NaN,NaN,60,87.0,79.0,WR,71.125,173,8.750,30.875,74.500,1.56,4.49,34.0,122.0,NaN,NaN,NaN,60,87.0,79.0,False
1,2023,2023,55904,3.909615,3.836488,6.630,6.281647,6.56450,6.185824,2.800476,2.744049,5.190,5.078353,0.00,0.165882,9.91,10.738118,9.890203,10.703258,109.88,121.196706,0.984030,0.959054,2.4,2.571765,3.162278,3.332181,8.0,9.720930,25.0,0.1,0.1,0,0,85,15,10,43,85,50.588235,100.0,11,12,5.863228,1.911702,8.083333,0.856667,8.025333,0.837250,3.372331,...,36.715069,20.286139,33.895591,21.149029,0.922073,0.188872,177.740645,65.661474

,draft_year,nfl_id,season,outer_fold
0,2023,55887,2023,1
1,2023,55887,2024,1
2,2023,55887,2025,1
3,2023,55904,2023,3
4,2023,55904,2024,3


## 1. Configuration and integrity checks

The core dataset is expected to contain 93 player-season observations from 54 receivers. If your local data differs, investigate the upstream sample construction rather than silently changing the expected sample.

In [3]:
KEYS = ["nfl_id", "season"]
OUTCOMES = [
    "game_mean_speed_raw_median",
    "game_peak_speed_raw_median",
    "game_p95_speed_raw_median",
    "game_mean_acceleration_raw_median",
    "game_peak_acceleration_raw_median",
    "game_path_efficiency_median",
]
TRADITIONAL = ["forty", "vertical", "broad_jump"]
TRACKING = [
    "combine_mean_speed_median_across_drills",
    "combine_peak_speed_median_across_drills",
    "combine_p95_speed_median_across_drills",
    "combine_mean_acceleration_median_across_drills",
    "combine_peak_acceleration_median_across_drills",
    "combine_path_efficiency_median_across_drills",
]
MODELS = {
    "TRADITIONAL": TRADITIONAL,
    "TRACKING": TRACKING,
    "COMBINED": TRADITIONAL + TRACKING,
}
ALPHAS = [0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000, 100000]
N_OUTER_SPLITS = 5
RANDOM_SEED = 2027
N_BOOTSTRAP = 5000

required = set(KEYS + ["draft_year"] + OUTCOMES + TRADITIONAL + TRACKING)
missing = sorted(required - set(df.columns))
assert not missing, f"Missing required dataset columns: {missing}"
assert not df.duplicated(KEYS).any(), "Duplicate player-season keys"
assert df["nfl_id"].notna().all() and df["season"].notna().all()
assert df["nfl_id"].nunique() == 54, f"Expected 54 players, found {df.nfl_id.nunique()}"
assert len(df) == 93, f"Expected 93 rows, found {len(df)}"

# Identify fold column names robustly.
fold_col = next((c for c in ["outer_fold", "fold", "outer_cv_fold"] if c in folds.columns), None)
assert fold_col is not None, f"No outer fold column found in {folds.columns.tolist()}"
assert "nfl_id" in folds.columns, "Fold assignments must include nfl_id"

# If season is present in fold table, join on player-season; otherwise join by player.
if "season" in folds.columns:
    fold_keys = folds[["nfl_id", "season", fold_col]].copy()
    assert not fold_keys.duplicated(KEYS).any()
    df = df.merge(fold_keys, on=KEYS, how="left", validate="one_to_one")
else:
    player_folds = folds[["nfl_id", fold_col]].drop_duplicates()
    assert not player_folds.duplicated(["nfl_id"]).any(), "A receiver has multiple outer fold assignments"
    df = df.merge(player_folds, on="nfl_id", how="left", validate="many_to_one")

assert df[fold_col].notna().all(), "Some player-seasons have no outer fold assignment"
assert df.groupby("nfl_id")[fold_col].nunique().max() == 1, "Receiver appears in multiple outer folds"
print("PASS — dataset integrity and receiver-grouped outer folds.")
print("Rows:", len(df), "| Players:", df.nfl_id.nunique(), "| Outer folds:", sorted(df[fold_col].unique()))

PASS — dataset integrity and receiver-grouped outer folds.
Rows: 93 | Players: 54 | Outer folds: [1, 2, 3, 4, 5]


## 2. Modeling functions

The inner loop selects alpha by mean validation MAE across receiver-grouped folds. Imputation and scaling are part of the pipeline, so they are fitted only on each training partition. The expected core sample is complete, but the imputer provides a defensive safeguard without leaking validation information.

In [4]:
def make_pipeline(alpha):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=float(alpha))),
    ])

def adaptive_group_splits(groups, requested=4):
    n_groups = pd.Series(groups).nunique()
    n_splits = min(requested, n_groups)
    if n_splits < 2:
        raise ValueError(f"Need at least 2 training receivers for inner CV; got {n_groups}")
    return n_splits

def tune_alpha(X, y, groups, alphas=ALPHAS):
    groups = np.asarray(groups)
    n_splits = adaptive_group_splits(groups, requested=4)
    cv = GroupKFold(n_splits=n_splits)
    scores = []
    for alpha in alphas:
        fold_maes = []
        for tr_idx, va_idx in cv.split(X, y, groups):
            model = make_pipeline(alpha)
            model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
            pred = model.predict(X.iloc[va_idx])
            fold_maes.append(mean_absolute_error(y.iloc[va_idx], pred))
        scores.append((float(alpha), float(np.mean(fold_maes))))
    score_df = pd.DataFrame(scores, columns=["alpha", "inner_mean_mae"])
    best = score_df.sort_values(["inner_mean_mae", "alpha"]).iloc[0]
    return float(best["alpha"]), score_df

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    rho = spearmanr(y_true, y_pred).statistic if len(y_true) >= 2 else np.nan
    return {
        "n": int(len(y_true)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)) if len(y_true) >= 2 else np.nan,
        "spearman": float(rho) if pd.notna(rho) else np.nan,
    }

def fit_outer_cv(data, outcome, model_name, predictors):
    prediction_rows, tuning_rows = [], []
    for fold_id in sorted(data[fold_col].unique()):
        train = data[data[fold_col] != fold_id].copy()
        test = data[data[fold_col] == fold_id].copy()

        train_players = set(train.nfl_id)
        test_players = set(test.nfl_id)
        assert train_players.isdisjoint(test_players), "Receiver leakage across outer fold"

        X_train, y_train = train[predictors], train[outcome].astype(float)
        X_test, y_test = test[predictors], test[outcome].astype(float)

        # Baseline uses only the outer training-fold median.
        baseline_value = float(y_train.median())
        baseline_pred = np.repeat(baseline_value, len(test))

        best_alpha, tuning = tune_alpha(X_train, y_train, train["nfl_id"].values)
        model = make_pipeline(best_alpha)
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        for i, (_, row) in enumerate(test.iterrows()):
            common = {
                "outcome": outcome, "model": model_name, "nfl_id": row["nfl_id"],
                "season": row["season"], "draft_year": row["draft_year"],
                "outer_fold": int(fold_id), "actual": float(y_test.iloc[i]),
            }
            prediction_rows.append({**common, "predicted": float(pred[i]), "best_alpha": best_alpha})
            if model_name == "TRADITIONAL":  # baseline stored once per outcome/fold
                prediction_rows.append({**common, "model": "BASELINE", "predicted": float(baseline_pred[i]), "best_alpha": np.nan})

        tuning_rows.append({
            "outcome": outcome, "model": model_name, "outer_fold": int(fold_id),
            "best_alpha": best_alpha, "inner_best_mae": float(tuning["inner_mean_mae"].min()),
            "train_rows": len(train), "test_rows": len(test),
            "train_players": train.nfl_id.nunique(), "test_players": test.nfl_id.nunique(),
            "alpha_at_grid_boundary": bool(best_alpha in [min(ALPHAS), max(ALPHAS)]),
        })
    return prediction_rows, tuning_rows

## 3. Run nested grouped cross-validation

Each receiver's player-season observations remain in one outer fold. The baseline prediction is calculated from the corresponding outer training partition. All three fitted models and the baseline are evaluated on the same held-out observations.

In [5]:
all_predictions, all_tuning = [], []
for outcome in OUTCOMES:
    print(f"Running outcome: {outcome}")
    outcome_data = df.dropna(subset=[outcome]).copy()
    assert outcome_data.nfl_id.nunique() >= 5
    for model_name, predictors in MODELS.items():
        # Common-sample assertion: the frozen core dataset should be complete for these predictors.
        model_data = outcome_data.dropna(subset=predictors).copy()
        assert set(model_data[KEYS].apply(tuple, axis=1)) == set(outcome_data[KEYS].apply(tuple, axis=1)), (
            f"{model_name} has a different sample for {outcome}; investigate missing predictors."
        )
        pred_rows, tune_rows = fit_outer_cv(model_data, outcome, model_name, predictors)
        all_predictions.extend(pred_rows)
        all_tuning.extend(tune_rows)

predictions = pd.DataFrame(all_predictions)
tuning_results = pd.DataFrame(all_tuning)

# No duplicate predictions for a player-season/model/outcome.
assert not predictions.duplicated(["outcome", "model", "nfl_id", "season"]).any()
display(predictions.head())
display(tuning_results.head())
print("Prediction rows:", len(predictions))

Running outcome: game_mean_speed_raw_median
Running outcome: game_peak_speed_raw_median
Running outcome: game_p95_speed_raw_median
Running outcome: game_mean_acceleration_raw_median
Running outcome: game_peak_acceleration_raw_median
Running outcome: game_path_efficiency_median


,outcome,model,nfl_id,season,draft_year,outer_fold,actual,predicted,best_alpha
0,game_mean_speed_raw_median,TRADITIONAL,55887,2023,2023,1,4.529600,4.153185,100.0
1,game_mean_speed_raw_median,BASELINE,55887,2023,2023,1,4.529600,4.163946,NaN
2,game_mean_speed_raw_median,TRADITIONAL,55938,2023,2023,1,4.604968,4.111945,100.0
3,game_mean_speed_raw_median,BASELINE,55938,2023,2023,1,4.604968,4.163946,NaN
4,game_mean_speed_raw_median,TRADITIONAL,56070,2023,2023,1,4.745906,4.107840,100.0


,outcome,model,outer_fold,best_alpha,inner_best_mae,train_rows,test_rows,train_players,test_players,alpha_at_grid_boundary
0,game_mean_speed_raw_median,TRADITIONAL,1,100.0,0.393284,74,19,43,11,False
1,game_mean_speed_raw_median,TRADITIONAL,2,100.0,0.396993,74,19,43,11,False
2,game_mean_speed_raw_median,TRADITIONAL,3,100.0,0.388015,74,19,43,11,False
3,game_mean_speed_raw_median,TRADITIONAL,4,100000.0,0.321494,75,18,44,10,True
4,game_mean_speed_raw_median,TRADITIONAL,5,1000.0,0.390056,75,18,43,11,False


Prediction rows: 2232


## 4. Aggregate out-of-fold performance

Metrics are computed from the pooled out-of-fold predictions. Interpret pooled R² cautiously because it combines held-out predictions from multiple folds; MAE and RMSE are the primary error metrics.

In [6]:
metric_rows = []
for (outcome, model_name), g in predictions.groupby(["outcome", "model"]):
    m = regression_metrics(g.actual, g.predicted)
    metric_rows.append({"outcome": outcome, "model": model_name, **m})
metrics = pd.DataFrame(metric_rows)
metric_order = ["BASELINE", "TRADITIONAL", "TRACKING", "COMBINED"]
metrics["model"] = pd.Categorical(metrics["model"], categories=metric_order, ordered=True)
metrics = metrics.sort_values(["outcome", "model"]).reset_index(drop=True)
display(metrics)

,outcome,model,n,mae,rmse,r2,spearman
0,game_mean_acceleration_raw_median,BASELINE,93,0.144469,0.183763,-0.033694,-0.232272
1,game_mean_acceleration_raw_median,TRADITIONAL,93,0.147980,0.188175,-0.083928,-0.142963
2,game_mean_acceleration_raw_median,TRACKING,93,0.151695,0.192080,-0.129386,-0.275726
3,game_mean_acceleration_raw_median,COMBINED,93,0.152166,0.193057,-0.140895,-0.206106
4,game_mean_speed_raw_median,BASELINE,93,0.387971,0.509557,-0.064864,-0.331299
5,game_mean_speed_raw_median,TRADITIONAL,93,0.395478,0.512723,-0.078138,-0.100602
6,game_mean_speed_raw_median,TRACKING,93,0.405508,0.527236,-0.140036,-0.223604
7,game_mean_speed_raw_median,COMBINED,93,0.395147,0.516501,-0.094088,-0.164864
8,game_p95_speed_raw_median,BASELINE,93,0.475230,0.649344,-0.111102,-0.325232
9,game_p95_speed_raw_median,TRADITIONAL,93,0.472152,0.619673,-0.011879,0.246282


## 5. Paired player-level bootstrap comparisons

For each outcome, compare model MAE with the baseline and compare Combined with Traditional. First calculate each receiver's mean absolute error across their player-season observations, then resample receivers with replacement. This preserves pairing and avoids treating repeated player-seasons as independent people.

A negative `mae_difference` means the first named model has lower MAE than the comparator.

In [7]:
def paired_player_bootstrap(pred_df, model_a, model_b, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED):
    a = pred_df[pred_df.model == model_a][["nfl_id", "season", "actual", "predicted"]].copy()
    b = pred_df[pred_df.model == model_b][["nfl_id", "season", "actual", "predicted"]].copy()
    a["ae_a"] = (a.actual - a.predicted).abs()
    b["ae_b"] = (b.actual - b.predicted).abs()
    paired = a[["nfl_id", "season", "ae_a"]].merge(
        b[["nfl_id", "season", "ae_b"]], on=["nfl_id", "season"], how="inner", validate="one_to_one"
    )
    per_player = paired.groupby("nfl_id")[["ae_a", "ae_b"]].mean()
    diffs = (per_player.ae_a - per_player.ae_b).to_numpy()
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot)
    for i in range(n_boot):
        sample = rng.choice(diffs, size=len(diffs), replace=True)
        boot[i] = sample.mean()
    return {
        "model_a": model_a, "model_b": model_b,
        "players": len(diffs), "player_mean_mae_a": float(per_player.ae_a.mean()),
        "player_mean_mae_b": float(per_player.ae_b.mean()),
        "mae_difference": float(diffs.mean()),
        "ci_2_5": float(np.quantile(boot, 0.025)),
        "ci_97_5": float(np.quantile(boot, 0.975)),
        "bootstrap_replicates": n_boot,
    }

bootstrap_rows = []
for outcome in OUTCOMES:
    g = predictions[predictions.outcome == outcome]
    for model_a, model_b in [
        ("TRADITIONAL", "BASELINE"),
        ("TRACKING", "BASELINE"),
        ("COMBINED", "BASELINE"),
        ("COMBINED", "TRADITIONAL"),
    ]:
        result = paired_player_bootstrap(g, model_a, model_b)
        bootstrap_rows.append({"outcome": outcome, **result})
bootstrap_results = pd.DataFrame(bootstrap_rows)
display(bootstrap_results)

,outcome,model_a,model_b,players,player_mean_mae_a,player_mean_mae_b,mae_difference,ci_2_5,ci_97_5,bootstrap_replicates
0,game_mean_speed_raw_median,TRADITIONAL,BASELINE,54,0.396385,0.380429,0.015956,-0.001714,0.033507,5000
1,game_mean_speed_raw_median,TRACKING,BASELINE,54,0.406031,0.380429,0.025602,0.008004,0.044432,5000
2,game_mean_speed_raw_median,COMBINED,BASELINE,54,0.397635,0.380429,0.017206,-0.002323,0.038089,5000
3,game_mean_speed_raw_median,COMBINED,TRADITIONAL,54,0.397635,0.396385,0.001250,-0.014201,0.016856,5000
4,game_peak_speed_raw_median,TRADITIONAL,BASELINE,54,0.469778,0.452546,0.017232,-0.023353,0.059560,5000
5,game_peak_speed_raw_median,TRACKING,BASELINE,54,0.474663,0.452546,0.022116,-0.010800,0.055227,5000
6,game_peak_speed_raw_median,COMBINED,BASELINE,54,0.447632,0.452546,-0.004915,-0.066010,0.056399,5000
7,game_peak_speed_raw_median,COMBINED,TRADITIONAL,54,0.447632,0.469778,-0.022146,-0.065348,0.020291,5000
8,game_p95_speed_raw_median,TRADITIONAL,BASELINE,54,0.474380,0.463281,0.011099,-0.033765,0.056834,5000
9,game_p95_speed_raw_median,TRACKING,BASELINE,54,0.482969,0.463281,0.019688,-0.011127,0.051820,5000


## 6. Leave-one-draft-cohort-out (LOCO)

Each draft cohort is held out in turn. Alpha is tuned using only the remaining draft cohorts and grouped inner CV. No player from the held-out cohort is used in training.

LOCO results are especially sensitive to small cohort sizes. Treat them as transportability checks, not definitive estimates of future-season performance.

In [8]:
def fit_loco(data, outcome, model_name, predictors):
    rows, tune_rows = [], []
    cohorts = sorted(data["draft_year"].dropna().unique())
    for cohort in cohorts:
        train = data[data.draft_year != cohort].copy()
        test = data[data.draft_year == cohort].copy()
        if train.nfl_id.nunique() < 5 or test.nfl_id.nunique() == 0:
            continue
        assert set(train.nfl_id).isdisjoint(set(test.nfl_id))
        y_train = train[outcome].astype(float)
        y_test = test[outcome].astype(float)
        baseline = float(y_train.median())
        best_alpha, tuning = tune_alpha(train[predictors], y_train, train.nfl_id.values)
        model = make_pipeline(best_alpha)
        model.fit(train[predictors], y_train)
        pred = model.predict(test[predictors])
        for i, (_, r) in enumerate(test.iterrows()):
            base = {
                "outcome": outcome, "model": model_name, "held_out_draft_year": int(cohort),
                "nfl_id": r.nfl_id, "season": r.season, "actual": float(y_test.iloc[i]),
            }
            rows.append({**base, "predicted": float(pred[i]), "best_alpha": best_alpha})
            if model_name == "TRADITIONAL":
                rows.append({**base, "model": "BASELINE", "predicted": baseline, "best_alpha": np.nan})
        tune_rows.append({
            "outcome": outcome, "model": model_name, "held_out_draft_year": int(cohort),
            "best_alpha": best_alpha, "inner_best_mae": float(tuning.inner_mean_mae.min()),
            "train_players": train.nfl_id.nunique(), "test_players": test.nfl_id.nunique(),
            "train_rows": len(train), "test_rows": len(test),
            "alpha_at_grid_boundary": bool(best_alpha in [min(ALPHAS), max(ALPHAS)]),
        })
    return rows, tune_rows

loco_predictions_rows, loco_tuning_rows = [], []
for outcome in OUTCOMES:
    for model_name, predictors in MODELS.items():
        r, t = fit_loco(df.dropna(subset=[outcome]).copy(), outcome, model_name, predictors)
        loco_predictions_rows.extend(r)
        loco_tuning_rows.extend(t)

loco_predictions = pd.DataFrame(loco_predictions_rows)
loco_tuning = pd.DataFrame(loco_tuning_rows)

loco_metric_rows = []
for (outcome, model_name, cohort), g in loco_predictions.groupby(["outcome", "model", "held_out_draft_year"]):
    loco_metric_rows.append({
        "outcome": outcome, "model": model_name, "held_out_draft_year": int(cohort),
        **regression_metrics(g.actual, g.predicted)
    })
loco_metrics = pd.DataFrame(loco_metric_rows)
display(loco_metrics)

,outcome,model,held_out_draft_year,n,mae,rmse,r2,spearman
0,game_mean_acceleration_raw_median,BASELINE,2023,45,0.169187,0.211680,-0.039524,NaN
1,game_mean_acceleration_raw_median,BASELINE,2024,34,0.108006,0.137716,-0.000482,NaN
2,game_mean_acceleration_raw_median,BASELINE,2025,14,0.150548,0.181506,-0.266422,NaN
3,game_mean_acceleration_raw_median,COMBINED,2023,45,0.184433,0.223885,-0.162856,0.034243
4,game_mean_acceleration_raw_median,COMBINED,2024,34,0.108406,0.137773,-0.001308,-0.050168
...,...,...,...,...,...,...,...,...
67,game_peak_speed_raw_median,TRACKING,2024,34,0.475014,0.629679,-0.083004,0.091778
68,game_peak_speed_raw_median,TRACKING,2025,14,0.371552,0.458166,-0.087634,0.116484
69,game_peak_speed_raw_median,TRADITIONAL,2023,45,0.467204,0.642607,-0.018833,0.577085
70,game_peak_speed_raw_median,TRADITIONAL,2024,34,0.504377,0.611023,-0.019779,0.175602


## 7. Diagnostics and interpretation safeguards

Check alpha grid boundaries, prediction dispersion, and model collapse. A model that predicts nearly the same value for everyone may have deceptively modest MAE but little ranking or explanatory usefulness.

In [9]:
diagnostic_rows = []
for (outcome, model_name), g in predictions.groupby(["outcome", "model"]):
    diagnostic_rows.append({
        "outcome": outcome, "model": model_name,
        "actual_sd": float(g.actual.std()),
        "prediction_sd": float(g.predicted.std()),
        "prediction_to_actual_sd_ratio": float(g.predicted.std() / g.actual.std()) if g.actual.std() > 0 else np.nan,
        "n_players": g.nfl_id.nunique(),
        "n_player_seasons": len(g),
    })
prediction_diagnostics = pd.DataFrame(diagnostic_rows)

alpha_diagnostics = tuning_results.groupby(["outcome", "model"]).agg(
    folds=("outer_fold", "nunique"),
    median_alpha=("best_alpha", "median"),
    min_alpha=("best_alpha", "min"),
    max_alpha=("best_alpha", "max"),
    boundary_alpha_folds=("alpha_at_grid_boundary", "sum"),
).reset_index()

display(prediction_diagnostics)
display(alpha_diagnostics)
display(loco_tuning)

,outcome,model,actual_sd,prediction_sd,prediction_to_actual_sd_ratio,n_players,n_player_seasons
0,game_mean_acceleration_raw_median,BASELINE,0.181723,0.011158,0.061400,54,93
1,game_mean_acceleration_raw_median,COMBINED,0.181723,0.042666,0.234785,54,93
2,game_mean_acceleration_raw_median,TRACKING,0.181723,0.040065,0.220474,54,93
3,game_mean_acceleration_raw_median,TRADITIONAL,0.181723,0.025168,0.138499,54,93
4,game_mean_speed_raw_median,BASELINE,0.496470,0.036580,0.073681,54,93
5,game_mean_speed_raw_median,COMBINED,0.496470,0.097854,0.197099,54,93
6,game_mean_speed_raw_median,TRACKING,0.496470,0.110109,0.221783,54,93
7,game_mean_speed_raw_median,TRADITIONAL,0.496470,0.086792,0.174818,54,93
8,game_p95_speed_raw_median,BASELINE,0.619364,0.074517,0.120312,54,93
9,game_p95_speed_raw_median,COMBINED,0.619364,0.285791,0.461426,54,93


,outcome,model,folds,median_alpha,min_alpha,max_alpha,boundary_alpha_folds
0,game_mean_acceleration_raw_median,COMBINED,5,100000.0,10.0,100000.0,3
1,game_mean_acceleration_raw_median,TRACKING,5,1000.0,10.0,100000.0,2
2,game_mean_acceleration_raw_median,TRADITIONAL,5,100000.0,0.1,100000.0,3
3,game_mean_speed_raw_median,COMBINED,5,1000.0,100.0,100000.0,1
4,game_mean_speed_raw_median,TRACKING,5,100000.0,10.0,100000.0,3
5,game_mean_speed_raw_median,TRADITIONAL,5,100.0,100.0,100000.0,1
6,game_p95_speed_raw_median,COMBINED,5,10.0,10.0,100.0,0
7,game_p95_speed_raw_median,TRACKING,5,1000.0,10.0,100000.0,1
8,game_p95_speed_raw_median,TRADITIONAL,5,10.0,10.0,100.0,0
9,game_path_efficiency_median,COMBINED,5,100000.0,10.0,100000.0,3


,outcome,model,held_out_draft_year,best_alpha,inner_best_mae,train_players,test_players,train_rows,test_rows,alpha_at_grid_boundary
0,game_mean_speed_raw_median,TRADITIONAL,2023,100000.0,0.372027,34,20,48,45,True
1,game_mean_speed_raw_median,TRADITIONAL,2024,100.0,0.374067,34,20,59,34,False
2,game_mean_speed_raw_median,TRADITIONAL,2025,100000.0,0.407273,40,14,79,14,True
3,game_mean_speed_raw_median,TRACKING,2023,100000.0,0.372038,34,20,48,45,True
4,game_mean_speed_raw_median,TRACKING,2024,100000.0,0.384217,34,20,59,34,True
5,game_mean_speed_raw_median,TRACKING,2025,10.0,0.397057,40,14,79,14,False
6,game_mean_speed_raw_median,COMBINED,2023,100000.0,0.372046,34,20,48,45,True
7,game_mean_speed_raw_median,COMBINED,2024,100.0,0.381167,34,20,59,34,False
8,game_mean_speed_raw_median,COMBINED,2025,10.0,0.388412,40,14,79,14,False
9,game_peak_speed_raw_median,TRADITIONAL,2023,100000.0,0.479837,34,20,48,45,True


## 8. Export all results

Outputs are saved under `reports/nested_ridge_modeling/`. These files form the audit trail for Notebook 08.

In [10]:
# Dataset fingerprint for reproducibility.
sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()

manifest = {
    "notebook": "08_nested_ridge_modeling.ipynb",
    "dataset": str(DATA_PATH.relative_to(PROJECT_ROOT)),
    "dataset_sha256": sha256,
    "rows": int(len(df)),
    "players": int(df.nfl_id.nunique()),
    "outcomes": OUTCOMES,
    "model_specifications": MODELS,
    "alphas": ALPHAS,
    "outer_cv": "5-fold GroupKFold grouped by nfl_id; assignments from Notebook 07",
    "inner_cv": "adaptive GroupKFold grouped by nfl_id; alpha selected by mean validation MAE",
    "bootstrap": {"unit": "receiver", "replicates": N_BOOTSTRAP, "seed": RANDOM_SEED},
    "loco": "leave one draft_year cohort out",
}

exports = {
    "oof_predictions.csv": predictions,
    "oof_predictions.parquet": predictions,
    "nested_cv_metrics.csv": metrics,
    "nested_cv_metrics.parquet": metrics,
    "inner_tuning_results.csv": tuning_results,
    "inner_tuning_results.parquet": tuning_results,
    "paired_player_bootstrap.csv": bootstrap_results,
    "paired_player_bootstrap.parquet": bootstrap_results,
    "prediction_diagnostics.csv": prediction_diagnostics,
    "alpha_diagnostics.csv": alpha_diagnostics,
    "loco_predictions.csv": loco_predictions,
    "loco_predictions.parquet": loco_predictions,
    "loco_metrics.csv": loco_metrics,
    "loco_metrics.parquet": loco_metrics,
    "loco_tuning_results.csv": loco_tuning,
}
for filename, table in exports.items():
    path = OUTPUT_DIR / filename
    if filename.endswith(".parquet"):
        table.to_parquet(path, index=False)
    else:
        table.to_csv(path, index=False)

(OUTPUT_DIR / "modeling_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("EXPORTED ARTIFACTS")
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(f"{path.name:42s} {path.stat().st_size:>12,} bytes")

EXPORTED ARTIFACTS
alpha_diagnostics.csv                             1,250 bytes
inner_tuning_results.csv                          7,926 bytes
inner_tuning_results.parquet                      8,510 bytes
loco_metrics.csv                                  8,929 bytes
loco_metrics.parquet                              8,378 bytes
loco_predictions.csv                            204,863 bytes
loco_predictions.parquet                         23,963 bytes
loco_tuning_results.csv                           4,988 bytes
modeling_manifest.json                            1,789 bytes
nested_cv_metrics.csv                             3,009 bytes
nested_cv_metrics.parquet                         6,108 bytes
oof_predictions.csv                             208,929 bytes
oof_predictions.parquet                          25,075 bytes
paired_player_bootstrap.csv                       3,987 bytes
paired_player_bootstrap.parquet                   8,410 bytes
prediction_diagnostics.csv                        2

## 9. Final validation

This final cell checks the expected model coverage and confirms that each fitted model generated one out-of-fold prediction per eligible player-season and outcome.

In [11]:
expected_fitted_models = {"TRADITIONAL", "TRACKING", "COMBINED"}
assert expected_fitted_models.issubset(set(predictions.model.unique()))
assert "BASELINE" in set(predictions.model.unique())
for outcome in OUTCOMES:
    g = predictions[predictions.outcome == outcome]
    counts = g.groupby("model").size()
    assert all(counts[m] == counts["TRADITIONAL"] for m in ["BASELINE", "TRACKING", "COMBINED"])
    assert g.groupby("model").nfl_id.nunique().nunique() == 1

assert not predictions[["actual", "predicted"]].isna().any().any()
assert not tuning_results["best_alpha"].isna().any()
print("=" * 78)
print("NOTEBOOK 08 NESTED RIDGE MODELING: PASS")
print("OOF prediction rows:", len(predictions))
print("Outcomes:", len(OUTCOMES))
print("Models:", sorted(predictions.model.astype(str).unique()))
print("Output directory:", OUTPUT_DIR)
print("=" * 78)

NOTEBOOK 08 NESTED RIDGE MODELING: PASS
OOF prediction rows: 2232
Outcomes: 6
Models: ['BASELINE', 'COMBINED', 'TRACKING', 'TRADITIONAL']
Output directory: D:\PROJECTS\WR-MovementIQ\reports\nested_ridge_modeling
